# RLHF Clinical Red-Teaming — CLI Driver
**Audrey Tjokro · Stephen Dong · Niki Karanikola**

This notebook is a **driver only**: no training logic, no hyperparameters, no model code lives here. It clones the repo at a pinned commit, installs deps, sets secrets, and shells out to `python -m src ...`.

Edit the **CONFIG** cell to choose method (`baseline` / `dpo` / `ppo`) and any overrides; everything else can be "Run all".

Per-run artifacts sync to `gs://results_043026/<method>/<run-uuid>/` at the end of the run.

## 1. Mount Drive (optional — used only as a local cache for HF + checkpoints)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone repo at a specific commit/branch
Pin a SHA for paper-grade reproducibility. Pass `BRANCH = 'main'` for HEAD.

In [9]:
REPO_URL = 'https://github.com/stephendongg/rlhf-clinical-redteaming.git'
BRANCH   = 'combined'
REPO_DIR = '/content/rlhf-clinical-redteaming'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())
%cd $REPO_DIR

203a600deab5fa116a120a88c9207d28cd3b6596
/content/rlhf-clinical-redteaming


In [26]:
!cd {REPO_DIR} && git pull origin combined


From https://github.com/stephendongg/rlhf-clinical-redteaming
 * branch            combined   -> FETCH_HEAD
Already up to date.


## 3. Install requirements

In [17]:
%pip install -q -r requirements.txt
%pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rlhf-clinical-redteaming (pyproject.toml) ... done


## 4. Secrets + GCS auth
Add `OPENAI_API_KEY` to Colab Secrets (left sidebar → key icon).

In [18]:
from google.colab import userdata, auth
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY').strip()

# GCS auth — uses your Google account; bucket gs://results_043026 must be in project rlhf-clinical-redteaming.
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = 'rlhf-clinical-redteaming'

## 5. CONFIG — pick method, run name, any overrides

In [27]:
METHOD     = 'dpo'
CONFIG     = f'configs/{METHOD}.yaml'
RUN_NAME   = 'dpo-smoketest'
GCS_BUCKET = 'gs://results_043026'

OVERRIDES = [
    'data.n_train=50',
    'data.n_dev=20',
    'dpo.n_per_seed=2',
    'judge_backend=hf_local',
    'judge_model=meta-llama/Llama-3.1-8B-Instruct',
    # Push utilization:
    'attacker_max_new_tokens=512',     # was 256 — longer generations
    'target_max_new_tokens=512',
    'lora.r=32',                       # was 16 — fatter adapter
    'lora.alpha=64',                   # keep alpha = 2*r
]

EXTRA_FLAGS = []

EXTRA_FLAGS = []


In [28]:
import shlex
cmd = ['python', '-m', 'src', METHOD,
       '--config', CONFIG,
       '--run-name', RUN_NAME,
       '--gcs-bucket', GCS_BUCKET]
for o in OVERRIDES:
    cmd += ['--override', o]
cmd += EXTRA_FLAGS
print('$', ' '.join(shlex.quote(c) for c in cmd))

$ python -m src dpo --config configs/dpo.yaml --run-name dpo-smoketest --gcs-bucket gs://results_043026 --override data.n_train=50 --override data.n_dev=20 --override dpo.n_per_seed=2 --override judge_backend=hf_local --override judge_model=meta-llama/Llama-3.1-8B-Instruct --override attacker_max_new_tokens=512 --override target_max_new_tokens=512 --override lora.r=32 --override lora.alpha=64


## 6. Run the CLI (artifacts sync to GCS at the end automatically)

In [29]:
import subprocess, sys
result = subprocess.run(cmd)
sys.exit(result.returncode) if result.returncode else print('OK')

SystemExit: 1

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 7. (Optional) Inspect this run's artifacts in GCS

In [24]:
!gsutil ls -r {GCS_BUCKET}/{METHOD}/ | tail -30

gs://results_043026/dpo/:

gs://results_043026/dpo/110a449d5808/:
gs://results_043026/dpo/110a449d5808/run_record.json
gs://results_043026/dpo/110a449d5808/split_fingerprint.jsonl
gs://results_043026/dpo/110a449d5808/training_log.jsonl

gs://results_043026/dpo/110a449d5808/checkpoints/:
gs://results_043026/dpo/110a449d5808/checkpoints/history.json

gs://results_043026/dpo/110a449d5808/checkpoints/iter_000/:
gs://results_043026/dpo/110a449d5808/checkpoints/iter_000/README.md
gs://results_043026/dpo/110a449d5808/checkpoints/iter_000/adapter_config.json
gs://results_043026/dpo/110a449d5808/checkpoints/iter_000/adapter_model.safetensors
gs://results_043026/dpo/110a449d5808/checkpoints/iter_000/trajectories.pkl
